# Social choice assignment project CAI

We will analyse the dataset 00073-00000002.cat regarding the 2017 categorization of French electoral candidates. We will do this by applying various social choice voting rules and construct & evaluate their social choice functions. 

The data is categorical, with a ranking within the categories (1, 0, -1) possibly representing approve, neutral, disapprove. Many candidates may be ranked in any category. So then, we must count their category as their preference ranking and cannot assume any ranking within a preference category. 

## Loading the Data into Python

In [24]:
import re

def load_cat(filepath):
    metadata = {}
    ballots = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Metadata
            if line.startswith('#'):
                if ':' in line:
                    key, value = line[1:].split(':', 1)
                    metadata[key.strip()] = value.strip()
                continue

            # Skip lines that don't look like ballots
            if ':' not in line:
                continue

            count_str, prefs_str = line.split(':', 1)
            count = int(count_str.strip())

            groups = re.findall(r'\{([^}]*)\}', prefs_str)
            ranking = [
                {int(x.strip()) for x in group.split(',') if x.strip()}
                for group in groups
            ]

            ballots.append({
                "count": count,
                "ranking": ranking
            })


    return metadata, ballots

In [25]:
metadata, ballots = load_cat("00073-00000002.cat")

total_voters = metadata.get("NUMBER VOTERS", "Unknown")
print(f"Total voters: {total_voters}")

alternatives = metadata.get("NUMBER ALTERNATIVES", "Unknown")
print(f"Number of alternatives: {alternatives}")

print(ballots[0])

Total voters: 13197
Number of alternatives: 11
{'count': 238, 'ranking': [{10, 6}, {1, 11, 9}, {2, 3, 4, 5, 7, 8}]}


The count defines our score value in the scoring vector, while the ranking (preferences) have been stores in the ranking attribute. 

# Plurality Rule

We choose the best candidate by the number of times they are number 1 in the preferences. Hence this is the scoring rule with the scoring vector s = (1, 0,0,.., 0) for the U (universe of alternatives in the preferences). However, we will only look at the best category, so hence we can look at how many times each alternative (candidate) is in the category 1.

In [26]:
def plurality_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            top_group = ranking[0]
            for candidate in top_group:
                scores[candidate] = scores.get(candidate, 0) + count
    max_score = max(scores.values())
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [27]:
plurality_winner_idx = plurality_rule_SCF(ballots)

for idx in plurality_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Jean-Luc Mélenchon


# Anti - Plurality Rule (Veto Rule)

For the anti-plurality rule we reward alternatives who are not last, with a scoring vector of (1,1,...,0). We can thus find out how many times each alternative falls in the final category and then award the ones who do this least with the winner title. 


In [ ]:
# anti_plurality_score_vector = [1 if i < alternatives else 0 for i in range(alternatives)]
def anti_plurality_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            bottom_group = ranking[-1]
            for candidate in bottom_group:
                scores[candidate] = scores.get(candidate, 0) + count
    min_score = min(scores.values())
    winners = [candidate for candidate, score in scores.items() if score == min_score]
    return winners

In [29]:
anti_plurality_winner_idx = anti_plurality_rule_SCF(ballots)
for idx in anti_plurality_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Benoît Hamon


# Borda's Rule

With Borda's rule we award the alternatives by their rank. Hence the scoring vector is s = (n-1, n-2, ..., 0). We have 3 categorical ranks, so we have (2, 1, 0) and we will award each alternative falling into a rank with the scoring[rank] value

In [30]:
# borda_score_vector = [2, 1, 0]
def borda_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        num_groups = len(ranking)
        for i, group in enumerate(ranking):
            points = num_groups - i - 1
            for candidate in group:
                scores[candidate] = scores.get(candidate, 0) + points * count
    max_score = max(scores.values())
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [31]:
borda_winner_idx = borda_rule_SCF(ballots)
for idx in borda_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Jean-Luc Mélenchon


So far, the 3 scoring rules do not finding the condorcet winner in some preference profiles. Hence we will consider a condorcet extension too. 

# Copeland's Rule

Copeland's rule awards an alternative for a pairwaise result, win, lose or tie. Here we will consider the ties within a category to count as well with a value between 0 and 1. 

In [89]:
tie_value = 0.5

def copeland_rule_SCF(ballots):
    scores = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        for i, group in enumerate(ranking):
            for candidate in group:
                scores[candidate] = scores.get(candidate, 0) + sum(len(ranking[j]) for j in range(i+1, len(ranking))) * count # how many wins
                scores[candidate] = scores.get(candidate, 0) - sum(len(ranking[j]) for j in range(i)) * count # how many losses
                scores[candidate] = scores.get(candidate, 0) + tie_value * count*(len(group)-1) # how many ties
    max_score = max(scores.values())
    print(scores)
    winners = [candidate for candidate, score in scores.items() if score == max_score]
    return winners

In [90]:
copeland_winner_idx = copeland_rule_SCF(ballots)
for idx in copeland_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

{10: 73204.5, 6: 70014.5, 1: 34027.0, 11: 51071.5, 9: 30065.5, 2: -2030.5, 3: 6545.0, 4: 1220.5, 5: -12485.0, 7: 20754.5, 8: -20377.5}
Jean-Luc Mélenchon


# Single transferable vote (STV)


We look at the alternative that is winning the least and remove. We continue this process till we have a winner.

In [ ]:
def single_transferable_vote_SCF(ballots):
    # Let us tally the 1st place (1st category) wins for each candidate
    first_place_counts = {}
    for ballot in ballots:
        count = ballot["count"]
        ranking = ballot["ranking"]
        if ranking:
            top_group = ranking[0]
            for candidate in top_group:
                first_place_counts[candidate] = first_place_counts.get(candidate, 0) + count
    

# Is there a Condorcet Winner

In [91]:
def condorcet_winner_SCF(ballots):
    candidates = set()
    for ballot in ballots:
        for group in ballot["ranking"]:
            candidates.update(group)

    candidates = sorted(candidates)
    pairwise_wins = {candidate: 0 for candidate in candidates}

    for i in range(len(candidates)):
        for j in range(i + 1, len(candidates)):
            candidate_i = candidates[i]
            candidate_j = candidates[j]
            votes_for_i = 0
            votes_for_j = 0

            for ballot in ballots:
                count = ballot["count"]
                ranking = ballot["ranking"]
                position_i = next((idx for idx, group in enumerate(ranking) if candidate_i in group), float('inf'))
                position_j = next((idx for idx, group in enumerate(ranking) if candidate_j in group), float('inf'))

                if position_i < position_j:
                    votes_for_i += count
                elif position_j < position_i:
                    votes_for_j += count

            if votes_for_i > votes_for_j:
                pairwise_wins[candidate_i] += 1
            elif votes_for_j > votes_for_i:
                pairwise_wins[candidate_j] += 1

    num_candidates = len(candidates)
    winners = [candidate for candidate, wins in pairwise_wins.items() if wins == num_candidates - 1]
    return winners

In [92]:
condorcet_winner_idx = condorcet_winner_SCF(ballots)
for idx in condorcet_winner_idx:
    print(metadata[f"ALTERNATIVE NAME {idx}"])

Jean-Luc Mélenchon
